In [0]:
# ============================================
# CELL 1: Import libraries
# ============================================

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

print("Bronze to Silver notebook started")

Bronze to Silver notebook started


In [0]:
storage_account = "healthcarestoragerev01"

storage_key = dbutils.secrets.get(
    scope="healthcare-scope",
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("Storage authentication configured")

Storage authentication configured


In [0]:
# ============================================
# CELL 2: Define Bronze and Silver paths
# ============================================

storage_account = "healthcarestoragerev01"

container = "input"

bronze_path = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net/bronze"
)

silver_path = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net/silver"
)

print("Bronze path:")
print(bronze_path)

print("Silver path:")
print(silver_path)

Bronze path:
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze
Silver path:
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver


In [0]:
# ============================================
# CELL 3: Check Bronze data
# ============================================

display(
    dbutils.fs.ls(bronze_path)
)

path,name,size,modificationTime
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/departments/,departments/,0,1788278227000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/encounters/,encounters/,0,1788278247000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/insurance_claim_data/,insurance_claim_data/,0,1788278253000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/patients/,patients/,0,1788278259000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/providers/,providers/,0,1788278264000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/bronze/transactions/,transactions/,0,1788278268000


In [0]:
# ============================================
# CELL 4: Create Silver folder
# ============================================

dbutils.fs.mkdirs(silver_path)

print("Silver folder created/verified")

Silver folder created/verified


In [0]:
# ============================================
# CELL 5: Read Departments from Bronze
# ============================================

departments_bronze = (
    spark.read
    .format("delta")
    .load(f"{bronze_path}/departments")
)

print("Departments Bronze data loaded")
print("Records:", departments_bronze.count())

display(departments_bronze.limit(5))

Departments Bronze data loaded
Records: 20


DeptID,Name,_bronze_loaded_at
DEPT001,Emergency,null
DEPT002,Cardiology,null
DEPT003,Neurology,null
DEPT004,Oncology,null
DEPT005,Pediatrics,null


In [0]:
# ============================================
# CELL 6: Clean Departments data
# ============================================

df_silver_departments = departments_bronze

for column_name, data_type in df_silver_departments.dtypes:
    if data_type == "string":
        df_silver_departments = df_silver_departments.withColumn(
            column_name,
            F.trim(F.col(column_name))
        )

df_silver_departments = (
    df_silver_departments
    .dropDuplicates()
    .withColumn("_silver_load_timestamp", F.current_timestamp())
)

print("Departments data cleaned")
print("Silver records:", df_silver_departments.count())

display(df_silver_departments.limit(5))

Departments data cleaned
Silver records: 20


DeptID,Name,_bronze_loaded_at,_silver_load_timestamp
DEPT013,Surgery,null,2026-09-03T15:02:25.372453Z
DEPT012,Pathology,null,2026-09-03T15:02:25.372453Z
DEPT008,Gastroenterology,null,2026-09-03T15:02:25.372453Z
DEPT016,Ophthalmology,null,2026-09-03T15:02:25.372453Z
DEPT018,Psychiatry,null,2026-09-03T15:02:25.372453Z


In [0]:
# ============================================
# CELL 7: Write Departments to Silver
# ============================================

departments_silver_path = f"{silver_path}/departments"

(
    df_silver_departments.write
    .format("delta")
    .mode("overwrite")
    .save(departments_silver_path)
)

print("Departments successfully written to Silver")

Departments successfully written to Silver


In [0]:
# ============================================
# CELL 8: Verify Silver data
# ============================================

silver_departments = (
    spark.read
    .format("delta")
    .load(departments_silver_path)
)

print("Silver Departments records:", silver_departments.count())

display(silver_departments.limit(5))

Silver Departments records: 20


DeptID,Name,_bronze_loaded_at,_silver_load_timestamp
DEPT013,Surgery,null,2026-09-03T15:02:27.492936Z
DEPT012,Pathology,null,2026-09-03T15:02:27.492936Z
DEPT008,Gastroenterology,null,2026-09-03T15:02:27.492936Z
DEPT016,Ophthalmology,null,2026-09-03T15:02:27.492936Z
DEPT018,Psychiatry,null,2026-09-03T15:02:27.492936Z


In [0]:
# ============================================
# CELL 9: Bronze → Silver for all tables
# ============================================

tables = [
    "departments",
    "encounters",
    "insurance_claim_data",
    "patients",
    "providers",
    "transactions"
]

for table_name in tables:

    print(f"Processing: {table_name}")

    # Read Bronze
    df = (
        spark.read
        .format("delta")
        .load(f"{bronze_path}/{table_name}")
    )

    # Trim string columns
    for column_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column_name,
                F.trim(F.col(column_name))
            )

    # Remove duplicate records
    df = df.dropDuplicates()

    # Add Silver load timestamp
    df = df.withColumn(
        "_silver_load_timestamp",
        F.current_timestamp()
    )

    # Write to Silver
    output_path = f"{silver_path}/{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(output_path)
    )

    print(
        f"Completed: {table_name} | Records: {df.count()}"
    )

print("====================================")
print("BRONZE TO SILVER COMPLETED")
print("====================================")

Processing: departments
Completed: departments | Records: 20
Processing: encounters
Completed: encounters | Records: 10000
Processing: insurance_claim_data
Completed: insurance_claim_data | Records: 10000
Processing: patients
Completed: patients | Records: 5000
Processing: providers
Completed: providers | Records: 25
Processing: transactions
Completed: transactions | Records: 10000
BRONZE TO SILVER COMPLETED


In [0]:
# ============================================
# CELL 10: Verify Silver layer
# ============================================

display(
    dbutils.fs.ls(silver_path)
)

path,name,size,modificationTime
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/departments/,departments/,0,1788280214000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/encounters/,encounters/,0,1788280281000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/insurance_claim_data/,insurance_claim_data/,0,1788280289000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/patients/,patients/,0,1788280296000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/providers/,providers/,0,1788280303000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/transactions/,transactions/,0,1788280310000
